# Loom Demo Notebook

## Part 2: Live Demo (3 queries)
## Part 3: Eval Results & Failure Analysis

---
## Setup

In [1]:
import sys
import os
import json

# Change to checkpoint_rag root and add src to path
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
sys.path.insert(0, 'src')

from dotenv import load_dotenv
load_dotenv()

from ingest import get_file_paths, create_langchain_documents, chunk_documents, embed_chunks
from retrieve import get_documents, format_docs, create_bm25_retriever
from generate import create_rag_chain, generate_answer, RAGResponse

In [2]:
# Build the pipeline
print("Loading and chunking documents...")
file_grouping = get_file_paths('./corpus')
docs = create_langchain_documents(file_grouping)
chunks = chunk_documents(docs)
print(f"Created {len(chunks)} chunks")

print("\nConnecting to vector store...")
vectorstore = embed_chunks(chunks)

print("Creating BM25 retriever...")
bm25_retriever = create_bm25_retriever(chunks)

print("Building RAG chain...")
rag_chain = create_rag_chain(vectorstore, bm25_retriever)

print("\nPipeline ready!")

Loading and chunking documents...
Created 196 chunks

Connecting to vector store...


/Users/raghavmanoharanjayanthi/Downloads/BASWE_AI_Learning_Roadmap/Module_1_LLMs_and_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating BM25 retriever...
Building RAG chain...

Pipeline ready!


---
# Part 2: Live Demo - 3 Queries

| Difficulty | Query | What to show |
|------------|-------|-------------|
| **Easy** | Pro plan cost | Clean retrieval, exact answer |
| **Medium** | SSO configuration | Tests hybrid search |
| **Hard** | Max workflow steps | Known failure case |

In [3]:
def run_query_with_details(query: str):
    """Run a query and show all the details for the demo."""
    print(f"QUERY: {query}")
    print("=" * 60)
    
    # Step 1: Retrieval
    print("\nSTEP 1: RETRIEVAL (Hybrid Search)")
    retrieved_docs = get_documents(query, vectorstore, bm25_retriever)
    print(f"Retrieved {len(retrieved_docs)} documents:\n")
    for i, doc in enumerate(retrieved_docs[:5], 1):
        source = doc.metadata.get('source', 'unknown')
        doc_id = doc.metadata.get('id', 'no-id')[:8]
        print(f"  {i}. [{doc_id}...] {source}")
        print(f"     FULL CONTENT:")
        print(f"     {'-' * 50}")
        print(f"     {doc.page_content}")
        print(f"     {'-' * 50}\n")
    
    # Step 2: Context formatting
    print("\nSTEP 2: CONTEXT SENT TO LLM")
    context = format_docs(retrieved_docs)
    print(f"Context length: {len(context)} characters")
    
    # Step 3: Generation
    print("\nSTEP 3: GENERATION")
    response = rag_chain.invoke(query)
    
    print(f"\nANSWER: {response.answer}")
    print(f"\nSOURCES CITED: {response.sources}")
    print(f"\nCONFIDENCE: {response.confidence}")
    
    return response, retrieved_docs

## Query 1: EASY - "How much does the Pro plan cost?"

**Expected:** $299 per month  
**Why easy:** Single doc, exact keyword match

In [4]:
response1, docs1 = run_query_with_details("How much does the Pro plan cost?")

QUERY: How much does the Pro plan cost?

STEP 1: RETRIEVAL (Hybrid Search)
Retrieved 6 documents:

  1. [0e638860...] 5_checkpoint_project/corpus/product-docs/billing/plans.md
     FULL CONTENT:
     --------------------------------------------------
     Plans and Pricing

Helix offers three plans: Starter, Pro, and Enterprise.

Starter — $49/month

Best for individuals automating personal or small-team tasks.

100 workflow runs per month

1 user

Basic integrations: Slack, Email, Webhooks, Google Sheets

Email support, 48-hour response SLA

Pro — $299/month

Best for growing teams that need more integrations and API access.

10,000 workflow runs per month

Up to 10 users

All standard integrations

API access (1000 req/min)

90-day run log retention

Email + chat support, 8-hour response SLA

Enterprise — Custom pricing

Best for organizations with security, compliance, or scale requirements.

Unlimited workflow runs

Unlimited users

All integrations + custom integrations on request

### Analysis for Easy Query
- Did it retrieve the right doc? (plans.md)
- Is the answer grounded in context?
- Did it cite the correct source?

## Query 2: MEDIUM - "How do I configure SSO?"

**Expected:** Navigate to Settings > Security > SSO  
**Why medium:** Needs to find the right runbook, tests hybrid search

In [5]:
response2, docs2 = run_query_with_details("How do I configure SSO?")

QUERY: How do I configure SSO?

STEP 1: RETRIEVAL (Hybrid Search)
Retrieved 6 documents:

  1. [eb43d393...] 5_checkpoint_project/corpus/product-docs/admin/sso.md
     FULL CONTENT:
     --------------------------------------------------
     Forcing SSO

To require all users to sign in via SSO (no password fallback), enable Enforce SSO in security settings. Existing password-based sessions are invalidated; users must re-authenticate via SSO.

Service accounts and SSO

Service accounts (used for API keys) bypass SSO. They authenticate via API key only.

Common issues

"User not found" after SSO redirect: The IdP is not sending the user's email in the NameID field. Adjust your IdP's NameID format.

Loop redirect to IdP: Browser cookies are blocked. Check third-party cookie settings.
     --------------------------------------------------

  2. [fb48daec...] 5_checkpoint_project/corpus/product-docs/admin/sso.md
     FULL CONTENT:
     --------------------------------------------------
  

### Analysis for Medium Query
- Did it retrieve the SSO doc?
- Is the answer specific enough?
- Does hybrid search help here? (SSO is a keyword)

## Query 3: HARD - "What's the maximum number of steps allowed in a single workflow?"

**Expected:** 100 steps  
**Why hard:** This is a KNOWN FAILURE from our eval (Q001)

In [6]:
response3, docs3 = run_query_with_details("What's the maximum number of steps allowed in a single workflow?")

QUERY: What's the maximum number of steps allowed in a single workflow?

STEP 1: RETRIEVAL (Hybrid Search)
Retrieved 7 documents:

  1. [95c0ae9d...] 5_checkpoint_project/corpus/product-docs/workflows/error-handling.md
     FULL CONTENT:
     --------------------------------------------------
     You can override the default retry settings on individual steps:

Retry count: 0 to 5

Backoff strategy: Fixed, Linear, or Exponential

Initial delay: 0s to 5min

Failure handlers

Configure a failure handler at the workflow level under Settings → Failure handler. The handler can:

Send a Slack or email notification

Trigger another workflow

Write to a Failures collection for later review

Idempotency

Because steps may retry, your integrations should be idempotent. For HTTP calls, include an Idempotency-Key header to avoid duplicate side effects.
     --------------------------------------------------

  2. [cc580d62...] 5_checkpoint_project/corpus/tickets/ticket-1234-workflow-timeout.html


### Analysis for Hard Query (Known Failure)
- Expected doc: `product-docs/workflows/creating-workflows.md`
- What was actually retrieved?
- **Root cause:** Retrieval failure - the right doc wasn't in top results
- **Why:** The query is about "steps" but the doc might use different terminology

---
# Part 3: Eval Results & Failure Analysis

In [7]:
# Load eval results
with open('evals/results/eval_results.json') as f:
    eval_results = json.load(f)

print("AGGREGATE SCORES")
print("=" * 40)
agg = eval_results['aggregate_scores']
thresholds = eval_results['thresholds']

print(f"Faithfulness:      {agg['faithfulness']:.3f}  (target: >=0.70) {'PASS' if thresholds['faithfulness']['passed'] else 'FAIL'}")
print(f"Answer Relevance:  {agg['answer_relevance']:.3f}")
print(f"Context Precision: {agg['context_precision']:.3f}  (target: >=0.60) {'PASS' if thresholds['context_precision']['passed'] else 'FAIL'}")
print(f"Correctness:       {agg['correctness']:.3f}")

AGGREGATE SCORES
Faithfulness:      0.929  (target: >=0.70) PASS
Answer Relevance:  0.978
Context Precision: 0.720  (target: >=0.60) PASS
Correctness:       0.829


In [8]:
# Find low-scoring queries
print("\nLOW-SCORING QUERIES")
print("=" * 60)

for r in eval_results['per_query_results']:
    faith = r['faithfulness']['score']
    prec = r['context_precision']['score']
    correct = r['correctness']['score']
    
    if faith < 0.7 or prec < 0.5 or correct < 0.5:
        print(f"\n{r['id']}: {r['query'][:55]}...")
        print(f"  Faith: {faith:.2f} | Prec: {prec:.2f} | Correct: {correct:.2f}")
        print(f"  Category: {r['category']} | Difficulty: {r['difficulty']}")


LOW-SCORING QUERIES

Q001: What's the maximum number of steps allowed in a single ...
  Faith: 0.33 | Prec: 0.00 | Correct: 0.00
  Category: workflows | Difficulty: easy

Q002: How long can a workflow run before it times out?...
  Faith: 1.00 | Prec: 0.00 | Correct: 1.00
  Category: workflows | Difficulty: easy

Q011: How long is the audit log retained?...
  Faith: 0.67 | Prec: 1.00 | Correct: 1.00
  Category: admin | Difficulty: easy

Q013: How long is a download URL for an export valid?...
  Faith: 1.00 | Prec: 0.33 | Correct: 1.00
  Category: data | Difficulty: easy

Q017: What does HTTP 401 mean from the Helix API?...
  Faith: 0.60 | Prec: 0.50 | Correct: 1.00
  Category: api | Difficulty: easy

Q020: What's the support response SLA on Pro?...
  Faith: 1.00 | Prec: 0.25 | Correct: 1.00
  Category: billing | Difficulty: easy

Q028: How do I export my data for compliance?...
  Faith: 0.75 | Prec: 0.42 | Correct: 0.00
  Category: data | Difficulty: medium

Q036: Why does my Salesforc

---
## Failure Case 1: Q001 - Retrieval Failure

**Query:** "What's the maximum number of steps allowed in a single workflow?"  
**Scores:** Faith: 0.33 | Prec: 0.00 | Correct: 0.00

In [9]:
# Analyze Q001
q001 = eval_results['per_query_results'][0]

print("FAILURE CASE 1: Q001")
print("=" * 60)
print(f"Query: {q001['query']}")
print(f"\nGround Truth: {q001['ground_truth']}")
print(f"\nGenerated Answer: {q001['generated_answer']}")

print("\nRETRIEVAL ANALYSIS:")
print(f"Expected source: {q001['context_precision']['expected']}")
print(f"Retrieved sources: {q001['context_precision']['retrieved'][:5]}")
print(f"Matched: {q001['context_precision']['matched']}")

print("\nROOT CAUSE: RETRIEVAL FAILURE")
print("The expected doc (creating-workflows.md) was NOT retrieved.")
print("Instead, we got error-handling.md, timeout tickets, loops.md, etc.")
print("\nWhy? The query asks about 'steps' but the doc might use different terminology.")

FAILURE CASE 1: Q001
Query: What's the maximum number of steps allowed in a single workflow?

Ground Truth: The maximum number of steps in a single Helix workflow is 100.

Generated Answer: The maximum number of steps allowed in a single workflow is not explicitly stated in the provided documents. However, it is mentioned that the total workflow runtime is capped at 30 minutes regardless of the number of steps. For specific limits on the number of steps, further information would be needed.

RETRIEVAL ANALYSIS:
Expected source: ['product-docs/workflows/creating-workflows.md']
Retrieved sources: ['product-docs/workflows/error-handling.md', 'tickets/ticket-1234-workflow-timeout.html', 'product-docs/workflows/loops.md', 'product-docs/billing/usage-and-overages.md', 'tickets/ticket-1212-webhook-not-firing.html']
Matched: []

ROOT CAUSE: RETRIEVAL FAILURE
The expected doc (creating-workflows.md) was NOT retrieved.
Instead, we got error-handling.md, timeout tickets, loops.md, etc.

Why? The 

In [28]:
creating_workflow_chunks = [c for c in chunks if 'creating-workflows.md' in c.metadata['source']]

In [29]:
len(creating_workflow_chunks)

2

In [31]:
# Find ALL chunks that contain "creating-workflows" anywhere in source
for c in chunks:
    if 'creating-workflows' in c.metadata.get('source', '').lower():
        print(f"Source: {c.metadata['source']}")
        print(f"Content length: {len(c.page_content)}")
        print(f"Preview: {c.page_content[:]}...")
        print("---")

Source: ./corpus/product-docs/workflows/creating-workflows.md
Content length: 875
Preview: Creating Workflows

Workflows are the core of Helix. This guide covers everything you need to know about building them.

Workflow structure

Every workflow has exactly one trigger and one or more steps. Steps run sequentially by default but can be branched, looped, or run in parallel.

Creating a new workflow

From the Workflows page, click New Workflow. You will be taken to the workflow builder.

Workflow limits

Maximum steps per workflow: 100

Maximum runtime per run: 30 minutes

Workflow versions retained: 50 (older versions are deleted automatically)

If your workflow exceeds these limits, the run will be terminated and marked as failed.

Naming conventions

While not required, we recommend descriptive workflow names with action verbs (e.g., "Post new leads to #sales" rather than "Slack workflow"). Names are searchable in the dashboard.

Activating workflows...
---
Source: ./corpus/product-d

In [ ]:
# retrieval failure
# options - increase k
# smaller chunks - this chunk didnt get picked up in similarity search
# query rewriting (What's the maximum number of steps allowed in a single workflow) -> isn't clear that this is during workflow creation
# reranking - assuming after fix that relevant chunk is retrieved

---
## Failure Case 2: Q028 - Generation Failure

**Query:** "How do I export my data for compliance?"  
**Scores:** Faith: 0.75 | Prec: 0.42 | Correct: 0.00

In [32]:
# Find Q028
q028 = next(r for r in eval_results['per_query_results'] if r['id'] == 'Q028')

print("FAILURE CASE 2: Q028")
print("=" * 60)
print(f"Query: {q028['query']}")
print(f"\nGround Truth: {q028['ground_truth']}")
print(f"\nGenerated Answer: {q028['generated_answer']}")

print("\nRETRIEVAL ANALYSIS:")
print(f"Expected sources: {q028['context_precision']['expected']}")
print(f"Matched: {q028['context_precision']['matched']}")
print(f"Precision score: {q028['context_precision']['score']:.2f}")

print("\nCORRECTNESS ANALYSIS:")
print(f"Key facts required: {q028['correctness']['key_facts_required']}")
print(f"Key facts found: {q028['correctness']['key_facts_found']}")
print(f"Key facts missing: {q028['correctness']['key_facts_missing']}")

print("\nROOT CAUSE: GENERATION/CONTEXT ISSUE")
print("Some relevant docs were retrieved (prec: 0.42), but the answer")
print("missed key facts. Either the context was incomplete or the LLM")
print("didn't extract the right information.")

FAILURE CASE 2: Q028
Query: How do I export my data for compliance?

Ground Truth: Initiate exports via Settings → Data → Export, choosing data type (workflows, runs, audit log, users), date range, and format (JSON or CSV). Exports are generated asynchronously and a signed download URL valid for 24 hours is emailed when ready. For very large exports, chunk by date range to parallelize. The API equivalent is POST /v2/exports.

Generated Answer: To export your data for compliance, you can initiate a Data Subject Access Request (DSAR) export for GDPR compliance. This involves following the privacy procedure and compiling the necessary data within 7 days, which includes profile data, workspace memberships, workflows authored, and audit log entries. If you need a complete data dump, you can also request a bulk export if your workspace has more than 10 million run records.

RETRIEVAL ANALYSIS:
Expected sources: ['tickets/ticket-1349-export-stuck.html', 'product-docs/data/exporting-data.md']


---
## Proposed Fix: Add Reranking

**Change:** Add a cross-encoder reranker after hybrid search

**Why:** 
- Hybrid search casts a wide net but doesn't always rank the best doc first
- A reranker scores each doc against the actual query
- Would push the most relevant doc to position 1

**How to measure:**
- Run the same 50-query eval with reranking enabled
- Compare context precision: expect 0.72 -> 0.80+
- Check if Q001 and Q028 improve

In [ ]:
# Example of what reranking would look like
print("PROPOSED FIX: RERANKING")
print("=" * 60)
print("""
from flashrank import Ranker

reranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

def get_documents_with_rerank(query, vectorstore, bm25_retriever, k=3):
    # Step 1: Hybrid search (cast wide net)
    vector_results = vectorstore.similarity_search(query, k=10)
    bm25_results = bm25_retriever.invoke(query)
    
    # Step 2: Deduplicate
    combined = deduplicate(vector_results + bm25_results)
    
    # Step 3: Rerank (narrow down)
    passages = [{"id": i, "text": doc.page_content} for i, doc in enumerate(combined)]
    reranked = reranker.rerank(query, passages, top_n=k)
    
    return [combined[r["id"]] for r in reranked]
""")

print("\nExpected impact:")
print("- Context Precision: 0.72 -> 0.80+")
print("- Fewer retrieval failures like Q001")
print("- Better ranking = better context for LLM")

---
## Summary for Loom

### Live Demo Results
1. **Easy (Pro plan cost):** Clean retrieval, exact answer
2. **Medium (SSO config):** Hybrid search helps with keyword "SSO"
3. **Hard (Max steps):** Retrieval failure - known weakness

### Eval Results
- **Faithfulness:** 0.929 PASS (target: 0.70)
- **Context Precision:** 0.720 PASS (target: 0.60)

### Failure Analysis
1. **Q001:** Retrieval failure - right doc not retrieved
2. **Q028:** Generation issue - context incomplete or LLM missed facts

### Proposed Fix
Add reranking -> measure context precision improvement on same 50 queries